# This is the implementation of the new code for the 2024 April 8 Eclipse

### Library Imports

In [1]:
import os
import numpy as np
import matplotlib as mpl
from matplotlib import pyplot as plt
import matplotlib.dates as mdates

from IPython.display import Audio, display, clear_output
import scipy
from scipy import signal
import eclipse_calculator
import datetime

import hf_tdoa_lib as tdoa
%matplotlib inline

tdoa.setup_plotting_style()

### find_maxes()
Function that allows us to find the beat notes which is the maximum frequency

In [2]:
def find_maxes(X, freq, minfreq, maxfreq, threshold):
    maxima_freqs = []
    if maxfreq == 0:
        maxfreq = 10000
    start = np.argwhere(freq >= minfreq)[0][0]
    end = np.argwhere(freq <= maxfreq)[-1:][0][0]
    maxima_indices = np.argwhere(X[start:end] >= threshold).flatten()
    maxima_freqs = freq[start:end][maxima_indices]
    return maxima_freqs


### find_starting_points()
The find starting point function allows us to auto-correlate a clean non-QRM'd file and enables us to find the starting points of the first chirp in a wave file *****CHANGE*******

In [3]:
def find_starting_points(sample,wav_list,plot_start):
    starting_times = []
    for fpath in wav_list:
        wav_env_x,wav_fs = tdoa.load_wav(fpath)
        wav_env_x=wav_env_x['x']
        correlation = signal.correlate(wav_env_x,sample,mode='same')

        
        corrmax = (np.argwhere(correlation==max(correlation[0:len(correlation)//2])))
        corrmax = (corrmax[0][0])/wav_fs
        starting_times.append(corrmax)
        if plot_start == True:
            tlim = (corrmax -3, corrmax + 5)
            tau = np.arange(len(wav_env_x))*(1/wav_fs)

            fig = plt.figure(figsize=(15,8))
            ax = fig.add_subplot(3,1,1)
            xx = np.arange(len(sample))*(1/wav_fs)#sampe singal fs
            ax.plot(xx,sample)
            # ax.plot([corrmax,corrmax],[-.5,.5])
            # ax.set_xlim([0,0.01])
            ax.set_title("Sample")
            ax.set_xlabel("time [seconds]")
            
            ax = fig.add_subplot(3,1,2)
            ax.plot(tau,wav_env_x)
            ax.plot([corrmax,corrmax],[-.5,.5])
            ax.set_xlim(tlim)
            ax.set_title(fpath)
            
            ax = fig.add_subplot(3,1,3)
            ax.plot(tau,correlation)
            # ax.plot([corrmax,corrmax],[-.5,.5])
            ax.axvline(corrmax, color = 'yellow')
            # ax.set_xlim(tlim)
            ax.set_title(fpath)
        
            fig.tight_layout()
            clear_output(wait=True)
            plt.show()
            # fig.savefig('start_time_plots\\'+fpath[:-4]+'.png',bbox_inches='tight')
            plt.close(fig)
            break
    return starting_times

In [4]:
def find_beat_notes(data_set,wav_list,starting_times,range_gates,plot_fft):
    all_beats = []
    for file_num,fpath in enumerate(wav_list):
        # Load and filter WAV file using library function
        wav_df, _ = tdoa.load_wav(fpath)
        wav_env, wav_fs = tdoa.filter(wav_df, low_pass_freq=250, high_pass_freq=10)
        
        peaks1 = []
        peaks2 = []
        peaks3 = []
        peaks4 = []
        for i in range(0,5):
            peaks1.append(starting_times[file_num]+.245*i + 1.9)   ##we determined the spacing between the chirps was 1.9
            peaks2.append(starting_times[file_num]+.245*4 + 0.6 + .245*i + 1.9)
            peaks3.append(starting_times[file_num] + range_gates[4] + 0.75*i)
            peaks4.append(starting_times[file_num] + range_gates[4] + 0.75*4 + (0.75*i) + 1.111)
        
        peaks = np.array([peaks3,peaks4]) ###added peaks3 maybe that makes a difference
        peaks=peaks.flatten() ##flatten the peaks
        
        maxes= []
        
        for peak_num,peak in enumerate(peaks): 
            tlim = (peak+range_gates[0],peak+range_gates[1])
            minfreq=range_gates[2]
            maxfreq=range_gates[3]
            
            # Use library function for FFT
            X_psd, freq = tdoa.chirp_fft(wav_env, tlim=tlim)
            
            # Find maximum using library function
            maximum_x, maximum_y, local_peaks_x, local_peaks_y = tdoa.find_max(freq, X_psd, minfreq, maxfreq)
            
            maxes.append(maximum_x)
            print(fpath, end = "\r")
        all_beats.append(np.array(maxes).flatten())
    return np.array(all_beats)

# Data Processing

In [5]:
# # Sorted loading to keep filenames/time alignment consistent across platforms
recording_dir = os.path.join('data.old','recordings')
directory_40m    = os.path.join(recording_dir,'40M_TX-WA5FRF_RX-N5DUP_318km')
directory_40mSam = os.path.join(recording_dir,'40mAB5YO')
directory_60mSam = os.path.join(recording_dir,'60mAB5YO')

def _load_sorted(directory):
    names = sorted([n for n in os.listdir(directory) if os.path.isfile(os.path.join(directory, n))])
    paths = [os.path.join(directory, n) for n in names]
    return paths, names

# Load sorted file lists
wavlist_40m, files_40m       = _load_sorted(directory_40m)
wavlist_40mSam, files_40mSam = _load_sorted(directory_40mSam)
wavlist_60mSam, files_60mSam = _load_sorted(directory_60mSam)

In [6]:
base_dir   = 'data'
sweep_rate = 10  # Hz/ms

chirp_dct  = {}
chirp_dct['N5DUP_40m'] = dct = {}
dct['data_dir']   = os.path.join('data','TX_WA5FRF_EL09nn-RX_N5DUP_EL09so-0009km-40m')
dct['sweep_rate'] = sweep_rate

chirp_dct['AB5YO_40m'] = dct = {}
dct['data_dir']   = os.path.join('data','TX_WA5FRF_EL09nn-RX_AB5YO_EL09so-0009km-40m')
dct['sweep_rate'] = sweep_rate

chirp_dct['AB5YO_60m'] = dct = {}
dct['data_dir']   = os.path.join('data','TX_WA5FRF_EL09nn-RX_AB5YO_EL09so-0009km-60m')
dct['sweep_rate'] = sweep_rate

# # Path to chirp template - used for finding chirp locations via cross-correlation
# template   = os.path.join('templates', 'N6RFM_10Hz_per_ms_template.wav')

# data_dir   = os.path.join(base_dir, data_set)
# wavlist    = tdoa.obtain_wav_list(data_dir)

In [7]:
# # Load sample templates (from original workflow)
# sample_dir = os.path.join('data.old','samples')

# sample_path  = os.path.join(sample_dir,'CorrelationV7')
# sample_env_x,sample_fs = tdoa.load_wav(sample_path+'.wav')
# sample_env_x=sample_env_x['x']
# sample_40m = sample_env_x

# sample_path  = os.path.join(sample_dir,'CorrelationV7')
# sample_env_x,sample_fs = tdoa.load_wav(sample_path+'.wav')
# sample_env_x=sample_env_x['x']
# sample_40mSam = sample_env_x

# sample_path  = os.path.join(sample_dir,'CorrelationV8')
# sample_env_x,sample_fs = tdoa.load_wav(sample_path+'.wav')
# sample_env_x=sample_env_x['x']
# sample_60mSam = sample_env_x

# # # Save sample templates to WAV files (for use in other workflows)
# # PRN_40m_template = sample_40m[0.3:0.4]
# # fname = os.path.join('templates','PRN_40m_template.wav')
# # scipy.io.wavfile.write(fname, sample_fs, PRN_40m_template.astype(np.float32))

# # PRN_60m_template = sample_60mSam[1:1.5]
# # fname = os.path.join('templates','PRN_60m_template.wav')
# # scipy.io.wavfile.write(fname, sample_fs, PRN_60m_template.astype(np.float32))

# start_times_40    = find_starting_points(sample_40m[0.3:0.4],wavlist_40m,plot_start=False)
# start_times_Sam40 = find_starting_points(sample_40mSam[0.3:0.4], wavlist_40mSam, plot_start= False)
# start_times_Sam60 = find_starting_points(sample_60mSam[1:1.5], wavlist_60mSam, plot_start= False)

In [ ]:
fname = os.path.join('templates','PRN_40m_template.wav')
tmpl,tmpl_fs = tdoa.load_wav(fname)
PRN_40m_template = tmpl['x']

fname = os.path.join('templates','PRN_60m_template.wav')
tmpl,tmpl_fs = tdoa.load_wav(fname)
PRN_60m_template = tmpl['x']

start_times_40    = find_starting_points(PRN_40m_template, wavlist_40m,plot_start=False)
start_times_Sam40 = find_starting_points(PRN_40m_template, wavlist_40mSam, plot_start= False)
start_times_Sam60 = find_starting_points(PRN_60m_template, wavlist_60mSam, plot_start= False)

NameError: name 'PRN_60m_template' is not defined

In [ ]:
year = 2024
month = 4
day = 8

fortyBand = []
fortyBand_times = []

fortymSamBand = []
fortymSamBand_times = []

sixtymSamBand = []
sixtymSamBand_times = []


# Times derived from sorted filename lists
for file in files_40m:
    if file[4:10] == 'EL09nn':
        hour = int(file[18:20])
        minute = int(file[20:22])
        fortyBand_times.append(datetime.datetime(year, month, day, hour, minute))

for file in files_40mSam:
    if file[0:4] == '2404':
        hour = int(file[7:9])
        minute = int(file[9:11])
        fortymSamBand_times.append(datetime.datetime(year, month, day, hour, minute))

for file in files_60mSam:
    if file[0:4] == '2404':
        hour = int(file[7:9])
        minute = int(file[9:11])
        sixtymSamBand_times.append(datetime.datetime(year, month, day, hour, minute))


Okay it is time to start the analysis, lets find the starting points

In [ ]:
# start_times_40    = find_starting_points(sample_40m[0.3:0.4],wavlist_40m,plot_start=False)
# start_times_Sam40 = find_starting_points(sample_40mSam[0.3:0.4], wavlist_40mSam, plot_start= False)
# start_times_Sam60 = find_starting_points(sample_60mSam[1:1.5], wavlist_60mSam, plot_start= False)

Find the beat note frequencies using range gating (we only need this for the 10 hz/ms) THE RANGE GATES NEED CHANGE

In [ ]:
fortym_range_gates  = [-0.4,0.5, 10, 3*10, 4.8]
beats_40m    = find_beat_notes('40m',wavlist_40m,start_times_40,fortym_range_gates,plot_fft=False)
beats_40mSam = find_beat_notes('40m',wavlist_40mSam,start_times_Sam40,fortym_range_gates,plot_fft=False)

sixtymSam_range_gates = [-0.1,0.25,1, 10 * 3, 4.1] ##originally 4.1, and -0.2,0.2
beats_60mSam = find_beat_notes('60m',wavlist_60mSam,start_times_Sam60,sixtymSam_range_gates,plot_fft=False)

What to do after the data is processed, we call the POSTprocessing

In [ ]:
# Calculate obscuration data
lat_AB =  29.57499
lon_AB = -98.88718

sTime = datetime.datetime(2024,4,8,13,0)
eTime = datetime.datetime(2024,4,8,23,0)
dt    = datetime.timedelta(minutes=1)

eclipse_times = [sTime]
while eclipse_times[-1] < eTime:
    eclipse_times.append(eclipse_times[-1]+dt)

obsc_AB = eclipse_calculator.eclipse_calc.calculate_obscuration(eclipse_times, lat_AB, lon_AB)
#####################
lat_N5 = 30.9
lon_N5 = -99.3

obsc_N5 = eclipse_calculator.calculate_obscuration(eclipse_times, lat_N5, lon_N5)
solarAB = eclipse_calculator.solarContext.solarTimeseries(sTime,eTime,lat_AB,lon_AB)
solarN5 = eclipse_calculator.solarContext.solarTimeseries(sTime,eTime,lat_N5,lon_N5)

In [ ]:
ManuallyDecodedN5DUP40m = [1.62, 1.59, 1.72, 1.88, 1.89, 1.82, 1.9, 1.96, 1.95, 1.89, 1.96, 1.94, 2.02, 1.89, 1.8, 1.82, 1.89, 1.67, 1.76, 1.83, 1.85, 1.75, 1.87, 1.75, 1.97, 1.98, 1.87, 2.04, 1.92, 1.85, 1.72, 1.92, 1.71]
ManuallyDecodedAB5YO40m = [1.72, 1.69, 1.80, 1.84, 1.86, 1.86, 1.94, 1.90, 1.88, 2.04, 2.04, 2.04, 2.08, 2.08, 2.02, 1.94, 1.92, 1.97, 1.97, 1.86, 1.90, 1.91, 1.90, 1.91, 1.82, 1.91, 1.84, 1.84, 1.76, 1.93, 1.97, 1.97, 1.99, 2.05, 2.01, 2.00, 1.99, 2.02, 2.02, 2.02, 2.01, 1.97, 1.96, 1.92, 1.93, 1.95, 1.91,1.90, 1.91, 1.89]
ManuallyDecodedAB5YO60m = [1.61, 1.52, 1.59, 1.54, 1.64, 1.61, 1.64, 1.67, 1.60, 1.67, 1.64, 1.72, 1.63, 1.82, 1.75, 1.65, 1.85, 2.00, 1.72, 1.80, 1.92, 1.85, 1.92, 2.00, 2.09, 1.96, 1.92, 1.85, 1.75, 1.89, 1.89, 1.89, 1.85, 1.82, 1.72, 1.65, 1.57, 1.65, 1.68, 1.77]

In [ ]:
AutoCorrelatedN5DUP40m = [1.681, 1.719, 1.473, np.NaN, 2.030, 1.879, 1.719, 2.106, np.NaN, 1.851, 1.785, 2.068, 1.898, 1.775, np.NaN, 1.719, np.NaN, 1.605, 1.804, np.NaN, 1.870, 1.860, 1.917, 2.002, np.NaN, 2.162, np.NaN, 2.068, 1.936, 1.974, 1.964, 1.700]
AutoCorrelatedAB5YO40m = [1.700, 1.728, 1.737, 1.804, 1.785, 1.870, 1.775, 1.879, 1.841, 1.898, 2.002, 1.804, 2.040, np.NaN, 2.030, 1.945, 2.011, 1.926, 1.879, 1.926, 1.841, 1.860, 1.974, 2.077, 1.766, 1.851, 1.775, np.NaN, 1.709, np.NaN, 1.898, np.NaN, 1.813, np.NaN, np.NaN, np.NaN, 1.889, np.NaN, 1.889, np.NaN, np.NaN, 2.040, np.NaN, 1.917, np.NaN, 1.945, np.NaN, 1.785, 1.79, np.NaN]
AutoCorrelatedAB5YO60m = [1.5625, 1.635, 1.635, 1.688, 1.510, 1.802, 1.583, 1.938, 1.635, 1.656, 1.781, 1.906, 1.875, 1.917, 1.635, 1.885, 1.698, 1.552, 1.760, 1.938, 1.979, 1.958, 2.104, 2.104, 1.896, 1.969, 1.771, 1.802, 1.885, 1.875, 1.740, 1.844, 1.792, 1.865, 1.948, 1.583, 1.563, 1.635, 1.615, 1.553]

IonosondeLayer_Height = [263, 258.2, 253.2, 246.7, 249.7, 254.8, 249.8, 253, 252, 265.7, 275.6, 279.7, 275.5, 277.7, 279.2, 270.8, 263.5, 265.8, 283.6, 286, 298.1, 295.5, 297.2, 292.7, 295.6, 299.7, 296.4, 293, 288.3, 284.5, 280.7, 285, 298.6, 287.8, 286.5, 279.3, 283.8, 296.8, 289.1, 300.3, 291.9, 288.9, 300.4, 295.8, 289.4, 292, 300.7, 290.9, 307.4, 304.7, 292.8, 302.3, 313.6, 308.7, 300, 291.7, 302.4, 303.7, 301, 306.2, 298.3, 284.9, 291.4, 293.8, 292.5, 302.4, 286.6, 285.9, 276.5, 288, 286.5, 278.9, 275.9, 270.6, 280.7, 276.1, 273.1, 275.7, 279.2, 299.5, 290.9, 285.8, 278.9, 281, 288.8, 279.9, 281, 277.4, 277, 271.5, 276.9]
Ionosonde_times = [
    "14:00:05", "14:05:05", "14:10:05", "14:15:05", "14:20:05", "14:25:05", "14:30:05", "14:35:05", "14:40:05", "14:45:05",
    "14:50:05", "14:55:05", "15:00:05", "15:05:05", "15:10:05", "15:15:05", "15:20:05", "15:25:05", "15:30:05", "15:35:05",
    "15:40:05", "15:45:05", "15:50:05", "15:55:05", "16:00:05", "16:05:05", "16:10:05", "16:15:05", "16:20:05", "16:25:05",
    "16:30:05", "16:35:05", "16:40:05", "16:45:05", "16:50:05", "16:55:05", "17:00:05", "17:05:05", "17:10:05", "17:15:05",
    "17:20:05", "17:25:05", "17:30:05", "17:35:05", "17:40:05", "17:45:05", "17:50:05", "17:55:05", "18:00:05", "18:05:05",
    "18:10:05", "18:15:05", "18:20:05", "18:25:05", "18:30:05", "18:35:05", "18:40:05", "18:45:05", "18:50:05", "18:55:05",
    "19:00:05", "19:05:05", "19:10:05", "19:15:05", "19:20:05", "19:25:05", "19:30:05", "19:35:05", "19:40:05", "19:45:05",
    "19:50:05", "19:55:05", "20:00:05", "20:05:05", "20:10:05", "20:15:05", "20:20:05", "20:25:05", "20:30:05", "20:35:05",
    "20:40:05", "20:45:05", "20:50:05", "20:55:05", "21:00:05", "21:05:05", "21:10:05", "21:15:05", "21:20:05", "21:25:05",
    "21:30:05"
]

# Convert strings to datetime objects
# Assuming the date is April 8, 2024
date_str = "2024-04-08"

# Convert each time string to datetime object
Ionosonde_datetimes = [datetime.datetime.strptime(date_str + " " + time_str, "%Y-%m-%d %H:%M:%S") for time_str in Ionosonde_times]
print(Ionosonde_datetimes)

## Figure 11

In [ ]:
def post_processing(beats,set_name,times,plot_TDOAs=False):
    sweep_rate = 10

    beat_averages = np.ndarray(len(beats))
    AB5YOList = []
    
##Sorts through the range of sweep rates, in this case only 1
    for i in range(0,len(beat_averages)):
        beat_averages[i] = np.median(beats[i])
    
    # avg_across = np.transpose(beat_averages)
    # TDOAs = np.ndarray((1,len(beats)))

    TDOAs = beat_averages/sweep_rate
    print(TDOAs)
    
    x_hop = TDOAs[(TDOAs < 1)]
    y_hop = TDOAs[(TDOAs > 3)]
    
    times = np.array(times)
    avs = None

## Now plots TDOA's 
    if plot_TDOAs:
        fig = plt.figure(figsize=(15,8))
        ax = fig.add_subplot(1,1,1)
        
        ax.scatter(times[np.where(TDOAs > 3)], y_hop, label = 'y_hop mode')
        ax.plot(times[np.where(TDOAs > 3)], y_hop, linewidth = 0.5)
        ax.scatter(times,TDOAs, label=sweep_rate)
        ax.plot(times,TDOAs,linewidth=.5)

        # ax.set_xlim(datetime(2024,4,8,11,00),datetime(2023,10,14,19))
        miny = 0
        maxy = 3
        ax.set_ylim(miny,maxy)
        
        ax3 = ax.twinx()
        ax3.yaxis.set_ticks_position("right")
        ax3.yaxis.set_label_position("right")
        # ax3.spines["center"].set_position(("axes"))
        ax3.set_frame_on(True)
        ax3.patch.set_visible(False)
        newlabels = np.arange(miny,maxy+.1,.25)
        ax3.set_yticks(np.arange(0,len(newlabels)))
        
        # ax3.set_yticklabels((300/2)*newlabels)
        
        ax3.set_yticklabels((150 * newlabels))
        
        ax3.ticklabel_format()
        ax3.set_ylabel('Layer Height [km]') 

        import matplotlib.dates as mdates
        myFmt = mdates.DateFormatter('%H:%M')
        ax.xaxis.set_major_formatter(myFmt)
        ax.xaxis.set_major_locator(mdates.MinuteLocator(interval=15))

        ax.set_ylabel('TDOA [ms]')
        ax.set_xlabel('time')
        ax.legend()
        ax.set_title(set_name)
        fig.autofmt_xdate()
        plt.tight_layout()
        plt.show()
        plt.close(fig)
    
    return TDOAs,avs

In [ ]:
def plot_hmF2(beats,set_name,times,plot_TDOAs=False, obscuration_times = None, obscuration_values = None, manual_TDOAs = None, correlated_TDOAS = None, solar_val = None, lat = None, lon = None):
    sweep_rate = 10

    beat_averages = np.ndarray(len(beats))
    for i in range(0,len(beat_averages)):
        beat_averages[i] = np.mean(beats[i][1:-1])

    TDOAs = beat_averages/sweep_rate
    avs = None

    if plot_TDOAs:
        order = np.argsort(times)
        times_sorted = [times[i] for i in order]

        if set_name == 'TX WA5FRF - RX N5DUP (317km) 7.2 MHz':
           layer_heights = (142 *TDOAs) + 36.1
           manual_layer_heights = (142 *np.array(manual_TDOAs[1:])) + 36.1
           correlated_layer_heights = (142 *np.array(correlated_TDOAS[:])) + 36.1     
        else:
           layer_heights = TDOAs*150
           manual_layer_heights = (150 *np.array(manual_TDOAs[1:]))
           correlated_layer_heights = (150 *np.array(correlated_TDOAS[1:]))

        layer_heights = np.array(layer_heights)[order]
        manual_layer_heights = np.array(manual_layer_heights)[order]
        correlated_layer_heights = np.array(correlated_layer_heights)[order]

        fig = plt.figure(figsize=(15,8))
        ax = fig.add_subplot(1,1,1)      
        ax.scatter(times_sorted,layer_heights)
        ax.plot(times_sorted,layer_heights, label= 'Automated Frequency Analysis')
        ax.scatter(times_sorted,manual_layer_heights)
        ax.plot(times_sorted,manual_layer_heights,label="Manual Period Analysis", ls = 'dotted')
        ax.scatter(times_sorted, correlated_layer_heights)
        ax.plot(times_sorted,correlated_layer_heights,label="Auto-Correlated Analysis", ls = 'dashdot')
        solar_val.overlaySolarElevation(ax)
        solar_val.overlayEclipse(ax)

        ax.scatter(Ionosonde_datetimes, IonosondeLayer_Height, color = 'purple')
        ax.plot(Ionosonde_datetimes,IonosondeLayer_Height,label="Austin Ionosonde", color = 'purple', ls = '--')
        ax.set_xlim(datetime.datetime(2024, 4, 8, 14),datetime.datetime(2024, 4, 8, 20, 30))

        myFmt = mdates.DateFormatter('%H:%M')
        ax.xaxis.set_major_formatter(myFmt)
        ax.xaxis.set_major_locator(mdates.MinuteLocator(interval=15))

        ax.set_ylabel('Layer Height [km]')
        ax.set_xlabel('Time [UTC]')
        ax.legend(loc = 'upper left')
        ax.set_title(set_name)
        fig.autofmt_xdate()

        plt.tight_layout()
        plt.show()
        plt.close(fig)
    
    return TDOAs,avs


In [ ]:
TDOAS_PAUL40, avs40   = plot_hmF2(beats_40m,   "TX WA5FRF - RX N5DUP (317km) 7.2 MHz", times= fortyBand_times,plot_TDOAs=True,manual_TDOAs = ManuallyDecodedN5DUP40m,obscuration_times = eclipse_times, obscuration_values=obsc_N5, correlated_TDOAS = AutoCorrelatedN5DUP40m, solar_val = solarN5, lat = lat_N5, lon = lon_N5)

## Figure 12

In [ ]:
def figure_12(dict_args1, dict_args2):
    """
    Calls post_processingNORMAL twice with two sets of arguments and plots two graphs labeled (a) and (b).
    Each dict_args should be a dictionary of arguments for post_processingNORMAL.
    """
    import numpy as np
    import matplotlib.pyplot as plt
    from datetime import datetime

    def sort_by_time(times, arrays):
        order = np.argsort(times)
        times_sorted = [times[i] for i in order]
        arrays_sorted = [np.array(arr)[order] for arr in arrays]
        return times_sorted, arrays_sorted

    TDOAs1, avs1 = plot_hmF2(**dict_args1, plot_TDOAs=False)
    TDOAs2, avs2 = plot_hmF2(**dict_args2, plot_TDOAs=False)

    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(15, 16))

    # ---------- Plot (a) ----------
    args = dict_args1
    times = args['times']
    set_name = args['set_name']
    manual_TDOAs = args['manual_TDOAs']
    correlated_TDOAS = args['correlated_TDOAS']
    solar_val = args['solar_val']

    if set_name == 'TX WA5FRF - RX N5DUP (317km) 7.2 MHz':
        layer_heights = (142 * TDOAs1) + 36.1
        manual_layer_heights = (142 * np.array(manual_TDOAs[1:])) + 36.1
        correlated_layer_heights = (142 * np.array(correlated_TDOAS[:])) + 36.1
    else:
        layer_heights = TDOAs1 * 150
        manual_layer_heights = (150 * np.array(manual_TDOAs[1:]))
        correlated_layer_heights = (150 * np.array(correlated_TDOAS[1:]))

    times_sorted, (layer_heights, manual_layer_heights, correlated_layer_heights) = sort_by_time(
        times, (layer_heights, manual_layer_heights, correlated_layer_heights)
    )

    ax1.plot(times_sorted, layer_heights, label='Automated Frequency Analysis')
    ax1.scatter(times_sorted, layer_heights)
    ax1.plot(times_sorted, manual_layer_heights, label="Manual Period Analysis", ls='dotted')
    ax1.scatter(times_sorted, manual_layer_heights)
    ax1.plot(times_sorted, correlated_layer_heights, label="Auto-Correlated Analysis", ls='dashdot')
    ax1.scatter(times_sorted, correlated_layer_heights)
    
    solar_val.overlaySolarElevation(ax1)
    solar_val.overlayEclipse(ax1)

    try:
        ax1.plot(Ionosonde_datetimes, IonosondeLayer_Height, label="Austin Ionosonde", color='purple', ls='--')
        ax1.scatter(Ionosonde_datetimes, IonosondeLayer_Height, color='purple')
    except Exception:
        pass

    ax1.set_title('(a) ' + set_name)
    ax1.set_ylabel('Layer Height [km]')
    ax1.set_xlabel('Time [UTC]')
    ax1.set_ylim(200, 350)
    ax1.legend(loc='upper left')
    ax1.set_xlim(datetime(2024, 4, 8, 14), datetime(2024, 4, 8, 20, 30))
    time_locator = mdates.MinuteLocator(interval=15)
    time_formatter = mdates.DateFormatter('%H:%M')
    ax1.xaxis.set_major_locator(time_locator)
    ax1.xaxis.set_major_formatter(time_formatter)
    for label in ax1.get_xticklabels():
        label.set_rotation(45)
        label.set_horizontalalignment('right')

    # ---------- Plot (b) ----------
    args = dict_args2
    times = args['times']
    set_name = args['set_name']
    manual_TDOAs = args['manual_TDOAs']
    correlated_TDOAS = args['correlated_TDOAS']
    solar_val = args['solar_val']

    if set_name == 'TX WA5FRF - RX N5DUP (317km) 7.2 MHz':
        layer_heights = (142 * TDOAs2) + 36.1
        manual_layer_heights = (142 * np.array(manual_TDOAs[1:])) + 36.1
        correlated_layer_heights = (142 * np.array(correlated_TDOAS[:])) + 36.1
    else:
        layer_heights = TDOAs2 * 150
        manual_layer_heights = (150 * np.array(manual_TDOAs[1:]))
        correlated_layer_heights = (150 * np.array(correlated_TDOAS[1:]))

    times_sorted, (layer_heights, manual_layer_heights, correlated_layer_heights) = sort_by_time(
        times, (layer_heights, manual_layer_heights, correlated_layer_heights)
    )

    ax2.plot(times_sorted, layer_heights, label='Automated Frequency Analysis')
    ax2.scatter(times_sorted, layer_heights)
    ax2.plot(times_sorted, manual_layer_heights, label="Manual Period Analysis", ls='dotted')
    ax2.scatter(times_sorted, manual_layer_heights)
    ax2.plot(times_sorted, correlated_layer_heights, label="Auto-Correlated Analysis", ls='dashdot')
    ax2.scatter(times_sorted, correlated_layer_heights)

    solar_val.overlaySolarElevation(ax2)
    solar_val.overlayEclipse(ax2)

    try:
        ax2.plot(Ionosonde_datetimes, IonosondeLayer_Height, label="Austin Ionosonde", color='purple', ls='--')
        ax2.scatter(Ionosonde_datetimes, IonosondeLayer_Height, color='purple')
    except Exception:
        pass

    ax2.set_title('(b) ' + set_name)
    ax2.set_ylabel('Layer Height [km]')
    ax2.set_xlabel('Time [UTC]')
    ax2.set_ylim(200, 350)
    ax2.set_xlim(datetime(2024, 4, 8, 14), datetime(2024, 4, 8, 20, 30))
    ax2.legend(loc='upper left')
    ax2.xaxis.set_major_locator(time_locator)
    ax2.xaxis.set_major_formatter(time_formatter)
    for label in ax2.get_xticklabels():
        label.set_rotation(45)
        label.set_horizontalalignment('right')

    plt.tight_layout()
    plt.subplots_adjust(hspace=0.3)
    plt.show()
    plt.close(fig)

    return (TDOAs1, avs1), (TDOAs2, avs2)


In [ ]:
result = figure_12(
    dict_args1={
        "beats": beats_40mSam,
        "set_name": "TX WA5FRF - RX AB5YO (8.77km) 7.2 MHz",
        "times": fortymSamBand_times,
        "manual_TDOAs": ManuallyDecodedAB5YO40m,
        "obscuration_times": eclipse_times,
        "obscuration_values": obsc_AB,
        "correlated_TDOAS": AutoCorrelatedAB5YO40m,
        "solar_val": solarAB,
        "lat": lat_AB,
        "lon": lon_AB
    },
    dict_args2={
        "beats": beats_60mSam,
        "set_name": "TX WA5FRF - RX AB5YO (8.77km) 5.3 MHz",
        "times": sixtymSamBand_times,
        "manual_TDOAs": ManuallyDecodedAB5YO60m,
        "obscuration_times": eclipse_times,
        "obscuration_values": obsc_AB,
        "correlated_TDOAS": AutoCorrelatedAB5YO60m,
        "solar_val": solarAB,
        "lat": lat_AB,
        "lon": lon_AB
    }
)